In [1]:
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(PROJECT_ROOT)

In [2]:
import pandas as pd
from utils import *
import missingno as msno

In [3]:
df_ = pd.read_csv('data/panel/CampusFile_WM_Cities.csv')

/var/folders/46/ky7ff4bj6874xpchl1bc9cqh0000gn/T/ipykernel_26166/3864280694.py:1: DtypeWarning: Columns (1,11,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ = pd.read_csv('data/panel/CampusFile_WM_Cities.csv')


In [4]:
df = df_.copy()

In [5]:
df = df.replace(to_replace='Other missing', value=np.nan)

In [6]:
df.haustier_erlaubt = df.haustier_erlaubt.replace(to_replace='Not specified', value='By arrangement')
df.haustier_erlaubt.value_counts()

By arrangement    3262396
No                 611865
Yes                216542
Name: haustier_erlaubt, dtype: int64

In [7]:
bef = [ 'bef2', 'bef10', 'bef9', 'bef8', 'bef7', 'bef6', 'bef5', 'bef4', 'bef3']
enya = ["bef2", "bef10", "bef9", "bef8", "bef7", "bef6", "bef5", "bef4", "bef3", "anbieter", "spell", "hits_gen", "click_schnellkontakte_gen", "click_weitersagen_gen", "click_url_gen", "liste_show_gen", "liste_match_gen", "adat"]
acelya = ["anzahletagen", "nebenraeume", "baujahr"]
johannes = ["heizkosten_in_wm_enthalten", "bauphase", "lieferung"]
ana = ["click_schnellkontakte", "hits", "click_customer", "click_weitersagen", "click_url", "liste_show", "liste_match", "laufzeittage"]
enya_2 = ["gid2019", "kid2019"]
enya_3 = ["nutzflaeche", "ev_kennwert", "immobilientyp", "ev_wwenthalten", "energieeffizienzklasse", "energieausweistyp", "bef1", "letzte_modernisierung", 'rollstuhlgerecht', 'ergg_1km']
variables_not_needed_for_rfc = ['nebenkosten', 'heizkosten'] 
columns_to_drop = enya + acelya + ana + bef + johannes + enya_2 + enya_3 + variables_not_needed_for_rfc
df = df.drop(columns=columns_to_drop)
df.shape

(4755938, 24)

In [8]:
df.etage = (df
            .etage
            .replace(to_replace='Implausible value', value=np.nan)
            .replace(to_replace='Variable for other types only', value=np.nan)
            .astype(float))
df.etage.isna().mean()

0.17286894824953564

In [9]:
df = df.dropna(subset=['plz', 'etage'])

In [10]:
show_missing_values(df)

,Column Name,Min,Max,n Unique,NaN count,NaN percentage,dtype
S. No.,,,,,,,
1,obid,483504,1925607441,3097056,0,0.0%,int64
2,plz,NaN,NaN,6208,0,0.0%,object
3,mietekalt,144.0,2690.0,96726,0,0.0%,float64
4,wohnflaeche,18.3,165.0,13861,0,0.0%,float64
5,etage,-1.0,45.0,47,0,0.0%,float64
6,zimmeranzahl,NaN,NaN,107,1,0.0%,object
7,schlafzimmer,NaN,NaN,11,1875987,48.021%,object
8,badezimmer,NaN,NaN,7,1156818,29.612%,object
9,aufzug,NaN,NaN,2,614652,15.734%,object


In [11]:
df.schlafzimmer = df.schlafzimmer.replace(to_replace='Implausible value', value=np.nan).astype(float)
df.schlafzimmer.isna().mean()

0.4802946378039163

In [12]:
df.badezimmer = df.badezimmer.replace(to_replace='Implausible value', value=np.nan).astype(float)
df.badezimmer.isna().mean()

0.29615138583745004

In [13]:
df['is_schlafzimmer_imputed'] = df.schlafzimmer.isna()

impute_map = {
    0.0: 1.0,
    1.0: 1.0,
    2.0: 2.0,
    3.0: 3.0,
    4.0: 4.0,
    5.0: 3.0
}
df.loc[df.schlafzimmer.isna(), 'schlafzimmer'] = df.loc[df.schlafzimmer.isna(), 'badezimmer'].map(impute_map)
df.schlafzimmer.isna().mean()

0.28572192840524574

In [14]:
df = df.dropna(subset=['badezimmer'])

In [15]:
df = df.dropna(subset='keller')
df.keller.isna().mean()

0.0

In [16]:
df = df.dropna(subset=['gaestewc'])
df.gaestewc.isna().mean()

0.0

In [17]:
df = df.dropna(subset=['aufzug', 'garten'])
df.aufzug.isna().mean()

0.0

In [18]:
df.kategorie_Wohnung = df.kategorie_Wohnung.fillna('Other')

In [19]:
df = df.dropna(subset=['balkon', 'einbaukueche', 'foerderung'])

In [20]:
show_missing_values(df[df.columns[df.isna().any()]])

,Column Name,Min,Max,n Unique,NaN count,NaN percentage,dtype
S. No.,,,,,,,
1,parkplatz,NaN,NaN,2,1654485,78.131%,object
2,haustier_erlaubt,NaN,NaN,3,358766,16.942%,object


In [21]:
df.loc[df.haustier_erlaubt.isna(), 'haustier_erlaubt'] = 'By arrangement'

In [22]:
show_missing_values(df[df.columns[df.isna().any()]])

,Column Name,Min,Max,n Unique,NaN count,NaN percentage,dtype
S. No.,,,,,,,
1,parkplatz,NaN,NaN,2,1654485,78.131%,object


In [23]:
df.parkplatz.value_counts()

Yes    450406
No      12699
Name: parkplatz, dtype: int64

In [24]:
df['is_parkplatz_imputed'] = df.parkplatz.isna()
df.parkplatz = df.parkplatz.fillna('No')

In [25]:
show_missing_values(df)

,Column Name,Min,Max,n Unique,NaN count,NaN percentage,dtype
S. No.,,,,,,,
1,obid,18948200,1925607441,1577083,0,0.0%,int64
2,plz,NaN,NaN,5126,0,0.0%,object
3,mietekalt,148.0,2690.0,76408,0,0.0%,float64
4,wohnflaeche,18.3,165.0,13535,0,0.0%,float64
5,etage,-1.0,45.0,47,0,0.0%,float64
6,zimmeranzahl,NaN,NaN,90,0,0.0%,object
7,schlafzimmer,0.0,8.0,9,0,0.0%,float64
8,badezimmer,0.0,5.0,6,0,0.0%,float64
9,aufzug,NaN,NaN,2,0,0.0%,object


In [26]:
df.columns

Index(['obid', 'plz', 'mietekalt', 'wohnflaeche', 'etage', 'zimmeranzahl',
       'schlafzimmer', 'badezimmer', 'aufzug', 'balkon', 'einbaukueche',
       'foerderung', 'gaestewc', 'garten', 'keller', 'parkplatz',
       'ausstattung', 'haustier_erlaubt', 'heizungsart', 'kategorie_Wohnung',
       'objektzustand', 'blid', 'edat', 'rent_sqm', 'is_schlafzimmer_imputed',
       'is_parkplatz_imputed'],
      dtype='object')

In [27]:
yes_no_columns = ['aufzug', 'balkon', 'einbaukueche', 'foerderung', 'gaestewc', 'garten', 'keller', 'parkplatz']

df[yes_no_columns] = df[yes_no_columns].replace({'Yes': 1, 'No': 0})

In [28]:
df['year'] = df['edat'].str[:4]
df['month'] = df['edat'].str[5:].astype(int)
df = df.drop(columns=['edat'])

In [32]:
column_types = {
    "obid": "int",
    "plz": "int",
    "mietekalt": "float",
    "wohnflaeche": "float",
    "etage": "int",
    "zimmeranzahl": "float",
    "schlafzimmer": "float",
    "badezimmer": "float",
    "aufzug": "int",
    "balkon": "int",
    "einbaukueche": "int",
    "foerderung": "int",
    "gaestewc": "int",
    "garten": "int",
    "keller": "int",
    "parkplatz": "int",
    "ausstattung": "object",
    "haustier_erlaubt": "object",
    "heizungsart": "object",
    "kategorie_Wohnung": "object",
    "objektzustand": "object",
    "blid": "object",
    "rent_sqm": "float",
    "month": "int",
    "year": "int"
}

In [30]:
df.zimmeranzahl = df.zimmeranzahl.replace(to_replace='Implausible value', value=np.nan).astype(float).fillna(2.0)
df.zimmeranzahl.isna().mean()

0.0

In [33]:
df = df.astype(column_types)

In [ ]:
df.to_csv('data/panel/cleaned_data_2.csv', index=False)

In [34]:
df.columns

Index(['obid', 'plz', 'mietekalt', 'wohnflaeche', 'etage', 'zimmeranzahl',
       'schlafzimmer', 'badezimmer', 'aufzug', 'balkon', 'einbaukueche',
       'foerderung', 'gaestewc', 'garten', 'keller', 'parkplatz',
       'ausstattung', 'haustier_erlaubt', 'heizungsart', 'kategorie_Wohnung',
       'objektzustand', 'blid', 'rent_sqm', 'is_schlafzimmer_imputed',
       'is_parkplatz_imputed', 'year', 'month'],
      dtype='object')

In [35]:
df.to_pickle('data/panel/cleaned_data_2.pkl')

In [36]:
df.sample(100).to_csv('test.csv', index=False)